# STOIC 预处理：file1-file4 -> npy1-npy4

按 `colab_tcia_covid19_preprocess.ipynb` 同款思路处理 `.mha`（数据与输出全部在 Google Drive，不使用 GCS）：
- 每个体数据随机采样 3D patch（默认 50 个；z<128 时 33 个）
- patch 大小 128x128x128
- `Resize + ScaleIntensityRangePercentiles(1, 99.9)`
- 每个 block 分别处理 `file1`~`file4`，输出到 `npy1`~`npy4`

In [1]:
# 挂载 Google Drive（全流程只使用 Drive，不使用 GCS）
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Block 1: file1 -> npy1 (STOIC preprocess)
import os
import sys
import random
import subprocess
from pathlib import Path

import numpy as np

try:
    import SimpleITK as sitk
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'SimpleITK', '-q'])
    import SimpleITK as sitk

try:
    from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'monai', '-q'])
    from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles

PATCH_NUM = 50
PATCH_SIZE = (128, 128, 128)  # (z, y, x)
TAR_IMG_SIZE = (128, 128, 128)
SRC = Path('/content/drive/MyDrive/dataset/pretrain/STOIC/download/file1')
DST = Path('/content/drive/MyDrive/dataset/pretrain/STOIC/npy1')
DST.mkdir(parents=True, exist_ok=True)


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape  # [z, y, x]
    size_z, size_y, size_x = patch_size

    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1

    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2

    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2

    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2

    return img_array[z1:z2, y1:y2, x1:x2]


def preprocess_patch(vol_zyx):
    transforms = Compose([
        Resize(spatial_size=TAR_IMG_SIZE, mode='trilinear'),
        ScaleIntensityRangePercentiles(
            lower=1.0,
            upper=99.9,
            b_min=0.0,
            b_max=1.0,
            clip=True,
            relative=False,
            channel_wise=False,
        ),
    ])
    # MONAI expects [C, Z, Y, X]
    return transforms(np.expand_dims(vol_zyx, 0)).numpy()[0]


mha_files = sorted(SRC.glob('*.mha'))
print(f'source: {SRC}')
print(f'mha files: {len(mha_files)}')

saved = 0
missing = 0
failed = 0
for fp in mha_files:
    if not fp.exists():
        missing += 1
        print(f'missing file, skip: {fp.name}')
        continue

    try:
        image = sitk.GetArrayFromImage(sitk.ReadImage(str(fp)))  # [z, y, x]
    except Exception as e:
        failed += 1
        print(f'read failed, skip: {fp.name} | {e}')
        continue

    if image.ndim == 4:
        print(f'skip 4D: {fp.name}')
        continue

    patch_num = int(PATCH_NUM / 1.5) if image.shape[0] < PATCH_SIZE[0] else PATCH_NUM
    for i in range(patch_num):
        patch = cut_patch(image, PATCH_SIZE)
        patch = preprocess_patch(patch).astype(np.float32)
        out = DST / f'STOIC_{fp.stem}_0_{i+1}.npy'
        if out.exists():
            continue
        np.save(out, patch)
        saved += 1

print(f'new patches saved: {saved}')
print(f'missing files skipped: {missing}')q
print(f'read failures skipped: {failed}')
print(f'output: {DST}')

source: /content/drive/MyDrive/dataset/pretrain/STOIC/download/file1
mha files: 500
new patches saved: 24983
missing files skipped: 0
read failures skipped: 0
output: /content/drive/MyDrive/dataset/pretrain/STOIC/npy1


In [4]:
# Block 2: file2 -> npy2 (STOIC preprocess)
import os
import sys
import random
import subprocess
from pathlib import Path

import numpy as np

try:
    import SimpleITK as sitk
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'SimpleITK', '-q'])
    import SimpleITK as sitk

try:
    from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'monai', '-q'])
    from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles

PATCH_NUM = 50
PATCH_SIZE = (128, 128, 128)
TAR_IMG_SIZE = (128, 128, 128)
SRC = Path('/content/drive/MyDrive/dataset/pretrain/STOIC/download/file2')
DST = Path('/content/drive/MyDrive/dataset/pretrain/STOIC/npy2')
DST.mkdir(parents=True, exist_ok=True)


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size

    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1

    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2

    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2

    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2

    return img_array[z1:z2, y1:y2, x1:x2]


def preprocess_patch(vol_zyx):
    transforms = Compose([
        Resize(spatial_size=TAR_IMG_SIZE, mode='trilinear'),
        ScaleIntensityRangePercentiles(
            lower=1.0,
            upper=99.9,
            b_min=0.0,
            b_max=1.0,
            clip=True,
            relative=False,
            channel_wise=False,
        ),
    ])
    return transforms(np.expand_dims(vol_zyx, 0)).numpy()[0]


mha_files = sorted(SRC.glob('*.mha'))
print(f'source: {SRC}')
print(f'mha files: {len(mha_files)}')

saved = 0
missing = 0
failed = 0
for fp in mha_files:
    if not fp.exists():
        missing += 1
        print(f'missing file, skip: {fp.name}')
        continue

    try:
        image = sitk.GetArrayFromImage(sitk.ReadImage(str(fp)))
    except Exception as e:
        failed += 1
        print(f'read failed, skip: {fp.name} | {e}')
        continue

    if image.ndim == 4:
        print(f'skip 4D: {fp.name}')
        continue

    patch_num = int(PATCH_NUM / 1.5) if image.shape[0] < PATCH_SIZE[0] else PATCH_NUM
    for i in range(patch_num):
        patch = cut_patch(image, PATCH_SIZE)
        patch = preprocess_patch(patch).astype(np.float32)
        out = DST / f'STOIC_{fp.stem}_0_{i+1}.npy'
        if out.exists():
            continue
        np.save(out, patch)
        saved += 1

print(f'new patches saved: {saved}')
print(f'missing files skipped: {missing}')
print(f'read failures skipped: {failed}')
print(f'output: {DST}')

source: /content/drive/MyDrive/dataset/pretrain/STOIC/download/file2
mha files: 500
new patches saved: 25000
missing files skipped: 0
read failures skipped: 0
output: /content/drive/MyDrive/dataset/pretrain/STOIC/npy2


In [5]:
# Block 3: file3 -> npy3 (STOIC preprocess)
import os
import sys
import random
import subprocess
from pathlib import Path

import numpy as np

try:
    import SimpleITK as sitk
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'SimpleITK', '-q'])
    import SimpleITK as sitk

try:
    from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'monai', '-q'])
    from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles

PATCH_NUM = 50
PATCH_SIZE = (128, 128, 128)
TAR_IMG_SIZE = (128, 128, 128)
SRC = Path('/content/drive/MyDrive/dataset/pretrain/STOIC/download/file3')
DST = Path('/content/drive/MyDrive/dataset/pretrain/STOIC/npy3')
DST.mkdir(parents=True, exist_ok=True)


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size

    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1

    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2

    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2

    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2

    return img_array[z1:z2, y1:y2, x1:x2]


def preprocess_patch(vol_zyx):
    transforms = Compose([
        Resize(spatial_size=TAR_IMG_SIZE, mode='trilinear'),
        ScaleIntensityRangePercentiles(
            lower=1.0,
            upper=99.9,
            b_min=0.0,
            b_max=1.0,
            clip=True,
            relative=False,
            channel_wise=False,
        ),
    ])
    return transforms(np.expand_dims(vol_zyx, 0)).numpy()[0]


mha_files = sorted(SRC.glob('*.mha'))
print(f'source: {SRC}')
print(f'mha files: {len(mha_files)}')

saved = 0
missing = 0
failed = 0
for fp in mha_files:
    if not fp.exists():
        missing += 1
        print(f'missing file, skip: {fp.name}')
        continue

    try:
        image = sitk.GetArrayFromImage(sitk.ReadImage(str(fp)))
    except Exception as e:
        failed += 1
        print(f'read failed, skip: {fp.name} | {e}')
        continue

    if image.ndim == 4:
        print(f'skip 4D: {fp.name}')
        continue

    patch_num = int(PATCH_NUM / 1.5) if image.shape[0] < PATCH_SIZE[0] else PATCH_NUM
    for i in range(patch_num):
        patch = cut_patch(image, PATCH_SIZE)
        patch = preprocess_patch(patch).astype(np.float32)
        out = DST / f'STOIC_{fp.stem}_0_{i+1}.npy'
        if out.exists():
            continue
        np.save(out, patch)
        saved += 1

print(f'new patches saved: {saved}')
print(f'missing files skipped: {missing}')
print(f'read failures skipped: {failed}')
print(f'output: {DST}')

source: /content/drive/MyDrive/dataset/pretrain/STOIC/download/file3
mha files: 500
new patches saved: 24983
missing files skipped: 0
read failures skipped: 0
output: /content/drive/MyDrive/dataset/pretrain/STOIC/npy3


In [ ]:
# Block 4: file4 -> npy4 (STOIC preprocess)
import os
import sys
import random
import subprocess
from pathlib import Path

import numpy as np

try:
    import SimpleITK as sitk
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'SimpleITK', '-q'])
    import SimpleITK as sitk

try:
    from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'monai', '-q'])
    from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles

PATCH_NUM = 50
PATCH_SIZE = (128, 128, 128)
TAR_IMG_SIZE = (128, 128, 128)
SRC = Path('/content/drive/MyDrive/dataset/pretrain/STOIC/download/file4')
DST = Path('/content/drive/MyDrive/dataset/pretrain/STOIC/npy4')
DST.mkdir(parents=True, exist_ok=True)


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size

    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1

    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2

    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2

    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2

    return img_array[z1:z2, y1:y2, x1:x2]


def preprocess_patch(vol_zyx):
    transforms = Compose([
        Resize(spatial_size=TAR_IMG_SIZE, mode='trilinear'),
        ScaleIntensityRangePercentiles(
            lower=1.0,
            upper=99.9,
            b_min=0.0,
            b_max=1.0,
            clip=True,
            relative=False,
            channel_wise=False,
        ),
    ])
    return transforms(np.expand_dims(vol_zyx, 0)).numpy()[0]


mha_files = sorted(SRC.glob('*.mha'))
print(f'source: {SRC}')
print(f'mha files: {len(mha_files)}')

saved = 0
missing = 0
failed = 0
for fp in mha_files:
    if not fp.exists():
        missing += 1
        print(f'missing file, skip: {fp.name}')
        continue

    try:
        image = sitk.GetArrayFromImage(sitk.ReadImage(str(fp)))
    except Exception as e:
        failed += 1
        print(f'read failed, skip: {fp.name} | {e}')
        continue

    if image.ndim == 4:
        print(f'skip 4D: {fp.name}')
        continue

    patch_num = int(PATCH_NUM / 1.5) if image.shape[0] < PATCH_SIZE[0] else PATCH_NUM
    for i in range(patch_num):
        patch = cut_patch(image, PATCH_SIZE)
        patch = preprocess_patch(patch).astype(np.float32)
        out = DST / f'STOIC_{fp.stem}_0_{i+1}.npy'
        if out.exists():
            continue
        np.save(out, patch)
        saved += 1

print(f'new patches saved: {saved}')
print(f'missing files skipped: {missing}')
print(f'read failures skipped: {failed}')
print(f'output: {DST}')